# Challenge — Attention, Fine-Tuning DistilBERT, Évaluation, Inférence Explicable

⚠️ **Ce notebook n'a pas été exécuté dans cet environnement** (pas d'accès réseau à `huggingface.co` dans mon bac à sable). Le code est correct et prêt à tourner sur Colab (idéalement avec un GPU : `Runtime > Change runtime type > GPU`). Exécute-le toi-même et remplace les commentaires `# TODO: vérifier après exécution` par tes observations réelles.

**Ce que j'ai corrigé par rapport à l'énoncé brut, et pourquoi** — voir les notes dans chaque section.


In [ ]:
# Installation (Colab)
!pip install -q transformers datasets evaluate accelerate matplotlib scikit-learn


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)


## 1. Chargement et inspection des données

On charge `tweet_eval` (config `sentiment`) : 3 classes — negative (0), neutral (1), positive (2).


In [ ]:
raw_datasets = load_dataset("tweet_eval", "sentiment")
print(raw_datasets)

# Distribution des classes par split
for split in raw_datasets:
    labels = raw_datasets[split]["label"]
    unique, counts = np.unique(labels, return_counts=True)
    print(f"{split} -> ", dict(zip(unique.tolist(), counts.tolist())))


In [ ]:
label_names = {0: "negative", 1: "neutral", 2: "positive"}

# On sauvegarde 2 exemples par label pour la visualisation d'attention plus tard
example_tweets = {}
for label_id, label_name in label_names.items():
    filtered = raw_datasets["train"].filter(lambda ex: ex["label"] == label_id)
    example_tweets[label_name] = [filtered[i]["text"] for i in range(2)]

for label_name, tweets in example_tweets.items():
    print(f"\n--- {label_name} ---")
    for t in tweets:
        print(" -", t)


## 2. Pipeline de tokenisation

⚠️ Note : `set_format("torch")` doit être appliqué **après** le `map`, et uniquement sur les colonnes utilisées par le modèle (`input_ids`, `attention_mask`, `labels`) — sinon la colonne `text` (string) fait planter le `DataCollator`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    tokenized["labels"] = examples["label"]
    return tokenized

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

# On mélange le train, puis on ne garde que les colonnes utiles au format torch
tokenized_datasets["train"] = tokenized_datasets["train"].shuffle(seed=42)

columns_to_keep = ["input_ids", "attention_mask", "labels"]
tokenized_datasets = tokenized_datasets.remove_columns(
    [c for c in tokenized_datasets["train"].column_names if c not in columns_to_keep]
)
tokenized_datasets.set_format("torch")

print(tokenized_datasets)


In [ ]:
# Sous-échantillonnage optionnel pour un premier essai rapide.
# ⚠️ L'énoncé ne le mentionne pas, mais l'entraînement complet (~45 600 exemples, 3 epochs, batch 32)
# est long sur un GPU Colab gratuit. Décommente pour un run de validation rapide avant le run complet.

QUICK_TEST = False  # passe à True pour un sous-échantillon de validation du pipeline

if QUICK_TEST:
    train_dataset = tokenized_datasets["train"].select(range(3000))
    eval_dataset = tokenized_datasets["validation"].select(range(1000))
else:
    train_dataset = tokenized_datasets["train"]
    eval_dataset = tokenized_datasets["validation"]

test_dataset = tokenized_datasets["test"]


## 3. Configuration du fine-tuning

⚠️ **Correction nécessaire par rapport à l'énoncé** : `load_best_model_at_end=True` exige que `eval_strategy` et `save_strategy` soient identiques (ici `"epoch"` pour les deux). L'énoncé ne le précise pas — c'est une erreur classique qui fait planter le `Trainer` sinon.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "macro_f1": f1["f1"]}

training_args = TrainingArguments(
    output_dir="./distilbert-tweet-sentiment",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",      # doit matcher save_strategy pour load_best_model_at_end
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)


In [ ]:
# ⚠️ Étape longue — attends-toi à plusieurs dizaines de minutes sur un GPU T4 Colab
# avec le dataset complet. Utilise QUICK_TEST=True ci-dessus pour valider le pipeline d'abord.
train_result = trainer.train()
print(train_result)


In [ ]:
# Sauvegarde du meilleur checkpoint + tokenizer, prêts à être réutilisés par une équipe
trainer.save_model("./distilbert-tweet-sentiment/best_model")
tokenizer.save_pretrained("./distilbert-tweet-sentiment/best_model")
print("Modèle et tokenizer sauvegardés dans ./distilbert-tweet-sentiment/best_model")


## 4. Évaluation et calibration

On évalue sur la validation, puis on récupère les scores softmax sur le split test pour l'histogramme de confiance.

⚠️ **Ce que l'énoncé conflate** : un histogramme brut des scores softmax maximum ne prouve pas la sur/sous-confiance du modèle. Il montre juste la distribution des probabilités prédites — pas si ces probabilités correspondent à l'exactitude réelle. Pour une vraie lecture de calibration, il faut un **diagramme de fiabilité** (accuracy réelle par bin de confiance, comparée à la diagonale idéale). Je fais les deux : l'histogramme demandé, et le diagramme de fiabilité en bonus, seul capable de vraiment étayer une affirmation du type "le modèle est surconfiant".


In [ ]:
val_metrics = trainer.evaluate(eval_dataset=eval_dataset)
print("Validation :", val_metrics)


In [ ]:
test_predictions = trainer.predict(test_dataset)
logits = test_predictions.predictions
true_labels = test_predictions.label_ids

# Softmax manuel (stable numériquement)
def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

probs = softmax(logits)
predicted_labels = np.argmax(probs, axis=-1)
confidence_scores = np.max(probs, axis=-1)  # score softmax de la classe prédite

test_acc = accuracy_metric.compute(predictions=predicted_labels, references=true_labels)
test_f1 = f1_metric.compute(predictions=predicted_labels, references=true_labels, average="macro")
print("Test accuracy :", test_acc)
print("Test macro F1 :", test_f1)


In [ ]:
# Histogramme des scores de confiance (bins de 0.1), tel que demandé par l'énoncé
plt.figure(figsize=(7, 4))
plt.hist(confidence_scores, bins=np.arange(0.0, 1.05, 0.1), edgecolor="black")
plt.xlabel("Score de confiance (softmax max)")
plt.ylabel("Nombre d'exemples")
plt.title("Distribution des scores de confiance — split test")
plt.show()

# TODO: vérifier après exécution — décris ici si la distribution est concentrée
# vers 1.0 (signe possible de surconfiance) ou plus étalée.


In [ ]:
# BONUS — diagramme de fiabilité (reliability diagram), seul outil qui permet
# de vraiment conclure sur la sur/sous-confiance, contrairement au simple histogramme.
bins = np.arange(0.0, 1.1, 0.1)
bin_indices = np.digitize(confidence_scores, bins) - 1

bin_accuracies = []
bin_confidences = []
bin_counts = []

for b in range(len(bins) - 1):
    mask = bin_indices == b
    count = mask.sum()
    if count > 0:
        acc = (predicted_labels[mask] == true_labels[mask]).mean()
        conf = confidence_scores[mask].mean()
    else:
        acc, conf = 0, 0
    bin_accuracies.append(acc)
    bin_confidences.append(conf)
    bin_counts.append(count)

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibration parfaite")
plt.bar(bins[:-1] + 0.05, bin_accuracies, width=0.09, edgecolor="black", label="Accuracy réelle par bin")
plt.xlabel("Confiance moyenne du bin")
plt.ylabel("Accuracy réelle")
plt.title("Diagramme de fiabilité (calibration)")
plt.legend()
plt.show()

# Si les barres sont systématiquement SOUS la diagonale -> le modèle est surconfiant
# (il annonce plus de confiance que sa précision réelle ne le justifie).
# TODO: vérifier après exécution.


## 5. Inspection de l'attention

⚠️ **Correction indispensable, absente de l'énoncé** : dans les versions récentes de `transformers`, le chemin d'attention par défaut (SDPA) ne retourne pas les poids d'attention — `outputs.attentions` serait `None` sans le forcer explicitement. Il faut charger le modèle avec `attn_implementation="eager"` (ou passer `output_attentions=True` ET s'assurer que le modèle le supporte).


In [ ]:
base_model = AutoModel.from_pretrained(
    "distilbert-base-uncased",
    attn_implementation="eager",   # nécessaire pour récupérer les poids d'attention
    output_attentions=True,
).to(device)
base_model.eval()

# On choisit un exemple sauvegardé à l'étape 1 (ex: un tweet negative)
example_sentence = example_tweets["negative"][0]
print("Phrase choisie :", example_sentence)

inputs = tokenizer(example_sentence, return_tensors="pt", truncation=True, max_length=128).to(device)
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = base_model(**inputs)

# outputs.attentions : tuple de (num_layers) tenseurs (batch, num_heads, seq_len, seq_len)
last_layer_attention = outputs.attentions[-1][0]        # (num_heads, seq_len, seq_len)
avg_attention = last_layer_attention.mean(dim=0)        # moyenne sur les têtes -> (seq_len, seq_len)

# Attention du token [CLS] (position 0) vers chaque token
cls_attention = avg_attention[0].cpu().numpy()

print(list(zip(tokens, cls_attention.round(3))))


In [ ]:
plt.figure(figsize=(10, 3))
plt.bar(range(len(tokens)), cls_attention)
plt.xticks(range(len(tokens)), tokens, rotation=75)
plt.ylabel("Attention depuis [CLS]")
plt.title(f"Attention [CLS] -> tokens — dernière couche, moyenne sur les têtes\n\"{example_sentence}\"")
plt.tight_layout()
plt.show()

# TODO: vérifier après exécution — identifie les 2-3 tokens qui reçoivent le plus
# d'attention et vérifie s'ils correspondent à des mots porteurs de sens
# (ex: "terrible", "hate", "worst") plutôt qu'à des tokens fonctionnels ([SEP], ponctuation).


## 6. Fonction de production `analyze_text()`

Fonction prête à être intégrée dans un outil support : renvoie label prédit, confiance, et tokens les plus contributeurs (via l'attention `[CLS]`, moyenne sur toutes les têtes de la dernière couche).


In [ ]:
def analyze_text(text: str, top_k: int = 5):
    """
    Retourne {label, confidence, highlighted_tokens} pour un texte donné,
    en utilisant le modèle fine-tuné (classification) + le modèle de base (attention).
    """
    # 1. Prédiction via le modèle fine-tuné
    clf_inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        clf_outputs = model(**clf_inputs)
    clf_probs = softmax(clf_outputs.logits.cpu().numpy())[0]
    predicted_id = int(np.argmax(clf_probs))
    confidence = float(clf_probs[predicted_id])

    # 2. Tokens contributeurs via l'attention [CLS] du modèle de base
    with torch.no_grad():
        attn_outputs = base_model(**clf_inputs)
    last_layer_attn = attn_outputs.attentions[-1][0].mean(dim=0)  # moyenne sur les têtes
    cls_attn = last_layer_attn[0].cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(clf_inputs["input_ids"][0])
    # On exclut les tokens spéciaux du classement des "tokens contributeurs"
    special_tokens = {tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token}
    scored_tokens = [
        (tok, float(score)) for tok, score in zip(tokens, cls_attn)
        if tok not in special_tokens
    ]
    top_tokens = sorted(scored_tokens, key=lambda x: x[1], reverse=True)[:top_k]

    return {
        "label": label_names[predicted_id],
        "confidence": round(confidence, 4),
        "highlighted_tokens": top_tokens,
    }


# Test rapide
result = analyze_text("This service is absolutely terrible, worst experience ever.")
print(result)


## Bilan sans complaisance

Trois choses à ne pas oublier avant de rendre ce notebook :

1. **Je n'ai rien exécuté** — tous les chiffres, graphiques et conclusions "TODO" doivent être remplis par toi après un vrai run sur Colab. Si tu rends ce notebook avec les commentaires `# TODO` encore présents, c'est que tu ne l'as pas fait tourner.
2. **Le diagramme de fiabilité n'était pas demandé explicitement** — je l'ai ajouté parce que l'histogramme seul ne permet pas de conclure sur la calibration, contrairement à ce que l'énoncé sous-entend ("calibration-style histogram"). Si un correcteur strict s'en tient à la lettre de l'énoncé, mentionne que tu es allé au-delà pour combler cette faille méthodologique — ne le passe pas sous silence comme si c'était superflu.
3. **`attn_implementation="eager"`** est une correction technique nécessaire à l'énoncé tel qu'écrit, pas une option de confort — sans elle, la section 5 plante purement et simplement selon la version de `transformers` installée.
